# Credit-risk data preparation

**Prediction question:** Using information available at application/origination, will an issued loan enter default within the following 12 months?

This notebook stops before modelling. It:

1. validates the Bondora extract;
2. constructs a fully observable 12-month binary target;
3. removes loans that lack a complete 12-month outcome window;
4. creates chronological train/validation/test splits;
5. reports repeat-borrower overlap without discarding valid later applications;
6. audits preparation quality using the training split only; and
7. fits imputation and categorical encoding on training data only.

Post-origination fields may define the historical answer (`y`) but are never predictors (`X`). Gender and age are retained only as fairness-audit metadata.


In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_PATH = Path(
    os.getenv("BONDORA_DATA_PATH", PROJECT_ROOT / "data" / "LoanData_Bondora.csv")
)

TARGET = "default_12m"
HORIZON_YEARS = 1
TRAIN_SHARE = 0.70
VALIDATION_SHARE = 0.15

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")


## 1. Load and validate the raw extract

The data path is portable: by default it expects `data/LoanData_Bondora.csv` from the project root. Set the `BONDORA_DATA_PATH` environment variable if the file lives elsewhere.

Only schema and integrity checks are performed before splitting. Feature distributions and data-dependent preparation decisions are inspected on the training split later.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Bondora CSV not found at {DATA_PATH}. "
        "Place it in data/ or set BONDORA_DATA_PATH."
    )

raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Raw shape: {raw.shape[0]:,} rows x {raw.shape[1]:,} columns")
raw.head()


In [ ]:
REQUIRED_COLUMNS = {
    "LoanId",
    "UserName",
    "LoanDate",
    "DefaultDate",
    "ReportAsOfEOD",
    "Status",
}

missing_required = sorted(REQUIRED_COLUMNS.difference(raw.columns))
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

integrity_summary = pd.Series(
    {
        "rows": len(raw),
        "columns": raw.shape[1],
        "exact_duplicate_rows": int(raw.duplicated().sum()),
        "duplicate_LoanId_values": int(raw["LoanId"].duplicated().sum()),
        "missing_borrower_ids": int(raw["UserName"].isna().sum()),
    },
    name="count",
)
integrity_summary


In [ ]:
DATE_COLUMNS = [
    "ReportAsOfEOD",
    "LoanDate",
    "DefaultDate",
    "ListedOnUTC",
    "LoanApplicationStartedDate",
]

for column in DATE_COLUMNS:
    if column in raw.columns:
        # utc=True safely normalizes any mixed timezone offsets.
        raw[column] = pd.to_datetime(raw[column], errors="coerce", utc=True).dt.tz_localize(None)

date_quality = pd.DataFrame(
    {
        "missing": raw[[c for c in DATE_COLUMNS if c in raw.columns]].isna().sum(),
        "minimum": raw[[c for c in DATE_COLUMNS if c in raw.columns]].min(),
        "maximum": raw[[c for c in DATE_COLUMNS if c in raw.columns]].max(),
    }
)
date_quality


## 2. Construct the 12-month target

For a loan issued on date `t`:

- first require the report to observe the loan through `t + 1 year`;
- set `y = 1` when `DefaultDate` falls in `[t, t + 1 year]`; and
- otherwise set `y = 0`.

Loans without a complete year of follow-up are excluded, even when an early default is already visible. Although that early positive is individually known, retaining recent positives while excluding recent loans that have not yet defaulted would bias the newest cohorts toward the positive class. Requiring equal follow-up makes the binary classes comparable.


In [ ]:
label_frame = raw.copy()
label_frame["horizon_end"] = label_frame["LoanDate"] + pd.DateOffset(years=HORIZON_YEARS)

valid_loan_date = label_frame["LoanDate"].notna()
valid_report_date = label_frame["ReportAsOfEOD"].notna()
valid_default_sequence = (
    label_frame["DefaultDate"].isna()
    | label_frame["DefaultDate"].ge(label_frame["LoanDate"])
)

default_within_horizon = (
    label_frame["DefaultDate"].notna()
    & label_frame["DefaultDate"].ge(label_frame["LoanDate"])
    & label_frame["DefaultDate"].le(label_frame["horizon_end"])
)
full_horizon_observed = (
    valid_report_date
    & label_frame["ReportAsOfEOD"].ge(label_frame["horizon_end"])
)

# Ordinary binary classification requires equal follow-up for both classes.
eligible = valid_loan_date & valid_default_sequence & full_horizon_observed

label_frame[TARGET] = default_within_horizon.astype("int8")
labelled = label_frame.loc[eligible].copy()

target_construction_summary = pd.Series(
    {
        "raw_loans": len(label_frame),
        "fully_observed_12m_loans": len(labelled),
        "defaults_within_12m": int(labelled[TARGET].sum()),
        "non_defaults_within_12m": int((labelled[TARGET] == 0).sum()),
        "excluded_incomplete_or_invalid": int((~eligible).sum()),
        "invalid_default_before_loan": int((~valid_default_sequence).sum()),
    },
    name="count",
)

print(f"Latest eligible origination: {labelled['LoanDate'].max():%Y-%m-%d}")
target_construction_summary


In [ ]:
# Status is useful only as a diagnostic. It is not the target and will not enter X.
status_target_audit = pd.crosstab(
    labelled["Status"],
    labelled[TARGET],
    rownames=["current Status"],
    colnames=["default within 12 months"],
    margins=True,
)
status_target_audit


## 3. Primary chronological splitting

A random row split can mix historical and future lending periods. The primary evaluation therefore uses chronological windows: 70% train, 15% validation and 15% test by origination date. `LoanDate` controls the evaluation order but is not included as a predictor.

Repeat borrowers are retained because the production use case includes valid future applications from returning customers. `UserName` is excluded from `X`, and overlap counts are reported below. A separate borrower-disjoint split can later be used as a stricter cold-start robustness test for completely unseen applicants.


In [ ]:
if labelled["UserName"].isna().any():
    raise ValueError("Borrower-overlap reporting requires non-missing UserName values.")

labelled = labelled.sort_values(["LoanDate", "LoanId"]).copy()
train_cutoff = labelled["LoanDate"].quantile(TRAIN_SHARE)
validation_cutoff = labelled["LoanDate"].quantile(TRAIN_SHARE + VALIDATION_SHARE)

split_frame = labelled.copy()
split_frame["split"] = np.select(
    [
        split_frame["LoanDate"].le(train_cutoff),
        split_frame["LoanDate"].le(validation_cutoff),
    ],
    ["train", "validation"],
    default="test",
)

splits = {
    name: split_frame.loc[split_frame["split"].eq(name)].copy()
    for name in ("train", "validation", "test")
}

borrower_split_counts = split_frame.groupby("UserName")["split"].nunique()
cross_boundary_borrowers = borrower_split_counts.gt(1)

print(f"Train cutoff:      {train_cutoff:%Y-%m-%d}")
print(f"Validation cutoff: {validation_cutoff:%Y-%m-%d}")
print(f"Borrowers appearing in multiple windows: {int(cross_boundary_borrowers.sum()):,}")
print("Valid later applications retained: no rows purged.")


In [ ]:
split_summary = pd.DataFrame(
    {
        name: {
            "rows": len(frame),
            "borrowers": frame["UserName"].nunique(),
            "start": frame["LoanDate"].min(),
            "end": frame["LoanDate"].max(),
            "default_rate": frame[TARGET].mean(),
        }
        for name, frame in splits.items()
    }
).T
split_summary


In [ ]:
train_users = set(splits["train"]["UserName"])
validation_users = set(splits["validation"]["UserName"])
test_users = set(splits["test"]["UserName"])

borrower_overlap = pd.Series(
    {
        "train_validation": len(train_users & validation_users),
        "train_test": len(train_users & test_users),
        "validation_test": len(validation_users & test_users),
    },
    name="shared_borrowers",
)

assert splits["train"]["LoanDate"].max() <= splits["validation"]["LoanDate"].min()
assert splits["validation"]["LoanDate"].max() <= splits["test"]["LoanDate"].min()

print("Temporal leakage check passed: split dates are ordered.")
borrower_overlap.to_frame()


## 4. Define the application-time feature boundary

A whitelist is safer than dropping known bad columns: a newly added post-origination field cannot silently enter the model.

Excluded from predictors:

- outcome/performance fields such as status, default dates, arrears, recoveries, balances and payments;
- Bondora's ratings, probability of default and expected-loss outputs;
- identifiers and dates retained only for splitting/auditing;
- finalized pricing/servicing fields such as funded amount, interest and monthly payment; and
- protected attributes (`Gender`, `Age`), which remain available for fairness analysis.


In [ ]:
INCOME_COMPONENT_COLUMNS = [
    "IncomeFromPrincipalEmployer",
    "IncomeFromPension",
    "IncomeFromFamilyAllowance",
    "IncomeFromSocialWelfare",
    "IncomeFromLeavePay",
    "IncomeFromChildSupport",
    "IncomeOther",
]

PREVIOUS_LOAN_COLUMNS = [
    "NoOfPreviousLoansBeforeLoan",
    "AmountOfPreviousLoansBeforeLoan",
    "PreviousRepaymentsBeforeLoan",
    "PreviousEarlyRepaymentsBefoleLoan",  # Name is misspelled in the source data.
    "PreviousEarlyRepaymentsCountBeforeLoan",
]

RAW_CANDIDATE_FEATURES = [
    "NewCreditCustomer",
    "ApplicationSignedHour",
    "ApplicationSignedWeekday",
    "VerificationType",
    "Country",
    "AppliedAmount",
    "LoanDuration",
    "UseOfLoan",
    "Education",
    "MaritalStatus",
    "NrOfDependants",
    "EmploymentStatus",
    "EmploymentDurationCurrentEmployer",
    "WorkExperience",
    "OccupationArea",
    "HomeOwnershipType",
    *INCOME_COMPONENT_COLUMNS,
    "IncomeTotal",
    "ExistingLiabilities",
    "LiabilitiesTotal",
    "RefinanceLiabilities",
    "DebtToIncome",
    "FreeCash",
    *PREVIOUS_LOAN_COLUMNS,
]

missing_candidates = sorted(set(RAW_CANDIDATE_FEATURES).difference(split_frame.columns))
if missing_candidates:
    raise ValueError(f"Expected applicant-time columns are missing: {missing_candidates}")

PROTECTED_COLUMNS = ["Gender", "Age"]
AUDIT_COLUMNS = ["LoanId", "UserName", "LoanDate", "Country", *PROTECTED_COLUMNS, TARGET, "split"]

assert TARGET not in RAW_CANDIDATE_FEATURES
assert set(PROTECTED_COLUMNS).isdisjoint(RAW_CANDIDATE_FEATURES)
print(f"Raw candidate features: {len(RAW_CANDIDATE_FEATURES)}")


## 5. Training-only preparation diagnostics

The following summaries are based only on the training window. Validation and test features remain unseen while preparation choices are examined.


In [ ]:
train_raw_features = splits["train"][RAW_CANDIDATE_FEATURES].copy()

train_missingness = (
    train_raw_features.isna().mean().mul(100).sort_values(ascending=False).rename("missing_pct")
)
train_missingness.to_frame().head(20)


In [ ]:
# The component fields are audited on training data only.
# They are omitted below because historical coverage is inconsistent; IncomeTotal is retained.
income_component_audit = pd.DataFrame(
    {
        "non_zero_pct": [
            pd.to_numeric(train_raw_features[c], errors="coerce").gt(0).mean() * 100
            for c in INCOME_COMPONENT_COLUMNS
        ],
        "missing_pct": [train_raw_features[c].isna().mean() * 100 for c in INCOME_COMPONENT_COLUMNS],
    },
    index=INCOME_COMPONENT_COLUMNS,
)
income_component_audit


In [ ]:
categorical_audit_columns = [
    "VerificationType",
    "Country",
    "UseOfLoan",
    "Education",
    "MaritalStatus",
    "EmploymentStatus",
    "EmploymentDurationCurrentEmployer",
    "OccupationArea",
    "HomeOwnershipType",
]

train_category_audit = pd.DataFrame(
    {
        "unique_non_null": train_raw_features[categorical_audit_columns].nunique(dropna=True),
        "missing_pct": train_raw_features[categorical_audit_columns].isna().mean() * 100,
    }
).sort_values("unique_non_null", ascending=False)
train_category_audit


## 6. Deterministic feature engineering

These transformations use only values from the same row, so there are no fitted population statistics. The function is nevertheless applied separately to each split.

- Hour and weekday become cyclical sine/cosine pairs.
- Work experience receives a genuine ordered mapping; employer duration stays categorical because `Retiree` is not a higher tenure level.
- Dependants are normalized numerically, with missingness retained.
- Zero/non-positive total income is treated as unreported and left for training-fitted imputation.
- Missing repayment-history values are set to zero only when the row explicitly says the borrower has zero previous loans.
- Sparse and historically inconsistent income components are removed rather than used for unreliable ratios.


In [ ]:
WORK_EXPERIENCE_MAP = {
    "LessThan2Years": 0,
    "2To5Years": 1,
    "5To10Years": 2,
    "10To15Years": 3,
    "15To25Years": 4,
    "MoreThan25Years": 5,
}

CATEGORICAL_FEATURES = [
    "VerificationType",
    "Country",
    "UseOfLoan",
    "Education",
    "MaritalStatus",
    "EmploymentStatus",
    "EmploymentDurationCurrentEmployer",
    "OccupationArea",
    "HomeOwnershipType",
]


def as_category(series: pd.Series) -> pd.Series:
    # Normalize mixed numeric/string category codes without inventing an order.
    return series.map(lambda value: str(value) if pd.notna(value) else np.nan).astype("object")


def engineer_application_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame[RAW_CANDIDATE_FEATURES].copy()

    customer_text = out["NewCreditCustomer"].astype("string").str.lower()
    out["NewCreditCustomer"] = customer_text.map(
        {"true": 1, "false": 0, "1": 1, "0": 0}
    ).astype("float64")

    hour = pd.to_numeric(out["ApplicationSignedHour"], errors="coerce")
    weekday = pd.to_numeric(out["ApplicationSignedWeekday"], errors="coerce")
    out["application_hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["application_hour_cos"] = np.cos(2 * np.pi * hour / 24)
    out["application_weekday_sin"] = np.sin(2 * np.pi * weekday / 7)
    out["application_weekday_cos"] = np.cos(2 * np.pi * weekday / 7)

    dependant_text = out["NrOfDependants"].astype("string").replace({"10Plus": "10"})
    out["dependants_missing"] = dependant_text.isna().astype("int8")
    out["NrOfDependants"] = pd.to_numeric(dependant_text, errors="coerce").clip(upper=10)

    out["work_experience_ord"] = out["WorkExperience"].map(WORK_EXPERIENCE_MAP)
    out["is_retiree"] = out["EmploymentDurationCurrentEmployer"].eq("Retiree").astype("int8")

    out["IncomeTotal"] = pd.to_numeric(out["IncomeTotal"], errors="coerce")
    out["income_unreported"] = out["IncomeTotal"].le(0) | out["IncomeTotal"].isna()
    out["income_unreported"] = out["income_unreported"].astype("int8")
    out["IncomeTotal"] = out["IncomeTotal"].mask(out["IncomeTotal"].le(0))

    previous_count = pd.to_numeric(out["NoOfPreviousLoansBeforeLoan"], errors="coerce")
    out["NoOfPreviousLoansBeforeLoan"] = previous_count
    out["has_previous_loans"] = previous_count.gt(0).astype("int8")

    for column in PREVIOUS_LOAN_COLUMNS[1:]:
        out[column] = pd.to_numeric(out[column], errors="coerce")
        first_loan_missing = previous_count.eq(0) & out[column].isna()
        out.loc[first_loan_missing, column] = 0

    numeric_passthrough = [
        "AppliedAmount",
        "LoanDuration",
        "ExistingLiabilities",
        "LiabilitiesTotal",
        "RefinanceLiabilities",
        "DebtToIncome",
        "FreeCash",
    ]
    for column in numeric_passthrough:
        out[column] = pd.to_numeric(out[column], errors="coerce")

    for column in CATEGORICAL_FEATURES:
        out[column] = as_category(out[column])

    return out.drop(
        columns=[
            "ApplicationSignedHour",
            "ApplicationSignedWeekday",
            "WorkExperience",
            *INCOME_COMPONENT_COLUMNS,
        ]
    )


In [ ]:
X_engineered = {
    name: engineer_application_features(frame)
    for name, frame in splits.items()
}

X_train = X_engineered["train"]
X_validation = X_engineered["validation"]
X_test = X_engineered["test"]

y_train = splits["train"][TARGET].astype("int8").copy()
y_validation = splits["validation"][TARGET].astype("int8").copy()
y_test = splits["test"][TARGET].astype("int8").copy()

fairness_audit = {
    name: frame[AUDIT_COLUMNS].copy()
    for name, frame in splits.items()
}

assert all(TARGET not in frame.columns for frame in X_engineered.values())
assert all(set(PROTECTED_COLUMNS).isdisjoint(frame.columns) for frame in X_engineered.values())

print(f"Engineered feature count before encoding: {X_train.shape[1]}")
X_train.head()


In [ ]:
engineered_feature_audit = pd.DataFrame(
    {
        "dtype": X_train.dtypes.astype(str),
        "missing_pct_train": X_train.isna().mean() * 100,
        "unique_train": X_train.nunique(dropna=True),
    }
).sort_values(["missing_pct_train", "unique_train"], ascending=[False, False])
engineered_feature_audit


## 7. Fit data-dependent preparation on training data only

Numerical medians and one-hot category vocabularies are learned from `X_train` only, then reused unchanged for validation and test data. Unknown validation/test categories are safely ignored.

WoE is intentionally not implemented here. It uses the target and therefore requires leakage-safe cross-fitting/tuning inside the later modelling workflow. Scaling is also deferred because it is model-specific.


In [ ]:
categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()
numeric_columns = X_train.select_dtypes(include="number").columns.tolist()

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ],
    remainder="drop",
)

# The only fit call in this notebook: it sees training rows only.
X_train_prepared = preprocessor.fit_transform(X_train)
X_validation_prepared = preprocessor.transform(X_validation)
X_test_prepared = preprocessor.transform(X_test)

prepared_feature_names = preprocessor.get_feature_names_out()

pd.DataFrame(
    {
        "rows": [
            X_train_prepared.shape[0],
            X_validation_prepared.shape[0],
            X_test_prepared.shape[0],
        ],
        "prepared_features": [
            X_train_prepared.shape[1],
            X_validation_prepared.shape[1],
            X_test_prepared.shape[1],
        ],
    },
    index=["train", "validation", "test"],
)


In [ ]:
def contains_nan(matrix) -> bool:
    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix)
    return bool(np.isnan(values).any())


assert X_train_prepared.shape[0] == len(y_train)
assert X_validation_prepared.shape[0] == len(y_validation)
assert X_test_prepared.shape[0] == len(y_test)
assert X_train_prepared.shape[1] == X_validation_prepared.shape[1] == X_test_prepared.shape[1]
assert len(prepared_feature_names) == X_train_prepared.shape[1]
assert not contains_nan(X_train_prepared)
assert not contains_nan(X_validation_prepared)
assert not contains_nan(X_test_prepared)

print("Preparation checks passed:")
print("- repeat-borrower overlap is measured and UserName is absent from X")
print("- chronological windows remain ordered")
print("- target and protected attributes are absent from X")
print("- train/validation/test have identical prepared columns")
print("- no missing values remain after training-fitted preparation")


## Handoff to feature analysis

The following objects are ready in memory:

- raw engineered splits: `X_train`, `X_validation`, `X_test`;
- labels: `y_train`, `y_validation`, `y_test`;
- fairness metadata: `fairness_audit`;
- training-fitted transformer: `preprocessor`; and
- encoded matrices: `X_train_prepared`, `X_validation_prepared`, `X_test_prepared`.

Run `02_feature_analysis.ipynb` next. It executes this notebook, then measures feature relationships and standalone predictiveness using the training split only. Model fitting, threshold selection and final test evaluation remain outside both notebooks.
